# Práctica · Bloque 2 — Calidad, EDA, ingeniería de características y PCA
**Unidad didáctica:** Diseño de Soluciones de Inteligencia Artificial (CERTUS) · Sesiones 3 y 4

## Caso: ¿qué clientes se irán de la operadora móvil?
Una operadora móvil peruana quiere anticipar qué clientes **se cambiarán de operador** el próximo mes (variable objetivo `se_fue`: 1 = se fue, 0 = se quedó). Te entregan el archivo **`clientes_telecom.csv`** con 1 530 filas exportadas del sistema.

Como en los datos reales, el archivo **trae errores**: duplicados, nulos, textos mal escritos, edades imposibles, montos mal digitados y una fecha guardada como texto. Nadie te dirá dónde están: encontrarlos es parte de la práctica.

## La pregunta de la práctica
> **¿Cuánto mejora un modelo cuando antes de entrenarlo aplicamos lo visto en las sesiones 3 y 4?**

Vas a entrenar **el mismo modelo** tres veces y comparar sus **matrices de confusión**:

| Escenario | Qué datos recibe el modelo |
|---|---|
| **A · Sin estrategias** | Las columnas numéricas tal como vienen. Los nulos se rellenan con 0. Nada más. |
| **B · Con estrategias** | Datos limpios + EDA + features nuevas + codificación + selección + estandarización. |
| **C · Estrategias + PCA** | Lo mismo que B, pero resumido con PCA en las componentes que conservan el 90 % de la varianza. |

**Regla de juego:** el modelo (regresión logística) y los clientes de prueba son los mismos en los tres escenarios. Lo único que cambia es **cómo preparas los datos**. Así, cualquier diferencia en la matriz de confusión se debe a tu trabajo con los datos.

## Qué entregas
1. Este cuaderno completo, que **corre de inicio a fin** (`Entorno de ejecución → Reiniciar y ejecutar todo`).
2. Todas las celdas **✍️ Responde** contestadas con datos (usa la plantilla: *hallazgo con dato + qué significa para el negocio + qué implica para el modelo*).
3. La tabla comparativa final de la sección 9 y tu conclusión.

**Cómo trabajar:** en las celdas de código, reemplaza cada `___` por lo que corresponda. Las pistas están en los comentarios `# TODO`.

## 0. Preparación

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score, precision_score, recall_score, f1_score

pd.set_option("display.max_columns", 40)
print("Listo")

### 0.1 Cargar el archivo
En Colab: ícono de carpeta (izquierda) → **Subir** → `clientes_telecom.csv`. También puedes leerlo desde tu Drive.

In [ ]:
clientes = pd.read_csv("clientes_telecom.csv")
print(clientes.shape)
clientes.head()

### 0.2 La columna `particion`
El archivo ya trae una columna `particion` con los valores `train` (1 230 filas) y `test` (300 clientes). **Los 300 clientes de prueba son los mismos para todos los equipos y en los tres escenarios**: así las matrices de confusión se pueden comparar.

Dos reglas importantes:
- `particion` **no es una variable** del modelo: solo indica en qué grupo está cada fila.
- Todo lo que **aprende** de los datos (medianas para rellenar nulos, límites del IQR, `fit` del escalador y del PCA) se calcula **solo con `train`**. Si usas `test`, estás "espiando" el examen (adelanto del bloque 3).

In [ ]:
print(clientes["particion"].value_counts())

## 1. La herramienta de evaluación: la matriz de confusión
Esta función entrena una **regresión logística** con los datos de entrenamiento, predice a los 300 clientes de prueba y muestra la **matriz de confusión**. **Ejecútala sin modificarla.**

```text
                         PREDICCIÓN DEL MODELO
                        Se queda (0)     Se va (1)
REALIDAD  Se queda (0)   VN  ✔            FP  ✘  falsa alarma: le damos un beneficio a quien no se iba
          Se va (1)      FN  ✘            VP  ✔  fuga detectada a tiempo
                         ↑ fuga NO detectada: el cliente se va y no hicimos nada (el error más caro)
```

| Métrica | Fórmula | Pregunta que responde |
|---|---|---|
| Exactitud (accuracy) | (VN + VP) / total | ¿Qué % de todos los clientes clasificó bien? |
| Precisión | VP / (VP + FP) | De los que el modelo marcó como "se va", ¿cuántos realmente se iban? |
| **Sensibilidad (recall)** | VP / (VP + FN) | De los que **realmente se fueron**, ¿a cuántos detectó? |
| F1 | equilibrio entre precisión y recall | Una sola nota que combina las dos |

Para la operadora, el error más caro es el **FN**: un cliente que se va sin que lo hayamos detectado. Por eso, además de la exactitud, miraremos sobre todo el **recall**.

In [ ]:
resultados = {}   # aquí se guardan las métricas de cada escenario
matrices = {}

def evaluar_modelo(X_train, y_train, X_test, y_test, nombre):
    modelo = LogisticRegression(max_iter=1000)
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)

    mc = confusion_matrix(y_test, pred)
    matrices[nombre] = mc
    resultados[nombre] = {
        "n_variables": X_train.shape[1],
        "VN": mc[0, 0], "FP": mc[0, 1], "FN": mc[1, 0], "VP": mc[1, 1],
        "exactitud": round(accuracy_score(y_test, pred), 3),
        "precision": round(precision_score(y_test, pred, zero_division=0), 3),
        "recall": round(recall_score(y_test, pred), 3),
        "f1": round(f1_score(y_test, pred), 3),
    }
    ConfusionMatrixDisplay(mc, display_labels=["Se queda (0)", "Se va (1)"]).plot(cmap="Blues", colorbar=False)
    plt.title(f"Matriz de confusión · {nombre}")
    plt.xlabel("Predicción"); plt.ylabel("Realidad"); plt.show()
    print(pd.Series(resultados[nombre]))
    return modelo

## 2. Escenario A · El modelo SIN estrategias (línea base)
Hacemos lo que haría alguien con prisa: tomar las columnas que ya son números, poner 0 donde falta un dato y entrenar. Sin revisar calidad, sin EDA, sin features, sin escalar. **Ejecuta sin modificar.**

In [ ]:
numericas_crudas = clientes.select_dtypes("number").columns.drop("se_fue").tolist()
print("Variables que usa el escenario A:", numericas_crudas)

X_A = clientes[numericas_crudas].fillna(0)
y = clientes["se_fue"]
es_train = clientes["particion"] == "train"

modelo_A = evaluar_modelo(X_A[es_train], y[es_train], X_A[~es_train], y[~es_train], "A · Sin estrategias")

### ✍️ Responde (escenario A)
1. Lee la matriz: de los clientes de prueba que **realmente se fueron**, ¿a cuántos detectó el modelo (VP) y a cuántos dejó escapar (FN)?
2. La exactitud parece buena. ¿Por qué podría ser engañosa? (Pista: ¿qué exactitud tendría un modelo que siempre dice "se queda"?)
3. ¿Qué columnas del archivo **no** entraron al modelo A? ¿Crees que alguna podría ser importante?
4. Si apareció un aviso `ConvergenceWarning`, ¿qué crees que significa? (Pista: lee el aviso completo.)

*Tu respuesta:*

## 3. Calidad de datos (sesión 3 · Tema 5)
### 3.1 Primer vistazo

In [ ]:
clientes.info()

In [ ]:
clientes.describe().round(1)

### ✍️ Responde
Con `info()` y `describe()`: ¿qué columnas tienen nulos? ¿Ves algún mínimo o máximo imposible? ¿Qué columna debería ser fecha y está como texto?

*Tu respuesta:*

### 3.2 Diccionario de datos
Completa la descripción de cada variable. El resto de la tabla se calcula solo.

In [ ]:
descripciones = {
    "id_cliente": "Código único del cliente",
    "edad": "Edad del cliente en años",
    "departamento": "___",                 # TODO
    "tipo_plan": "___",                    # TODO
    "metodo_pago": "___",                  # TODO
    "fecha_alta": "Fecha en que el cliente se afilió (dd/mm/aaaa)",
    "consumo_datos_mb": "___",             # TODO: consumo de datos móviles del último mes
    "minutos_llamadas": "___",             # TODO
    "recargas_mes": "Número de recargas en el mes (solo prepago)",
    "monto_factura": "___",                # TODO
    "velocidad_mbps": "Velocidad promedio de internet móvil (Mbps)",
    "caidas_red_mes": "Veces que el cliente se quedó sin señal en el mes",
    "llamadas_soporte": "___",             # TODO
    "satisfaccion": "Nota de la última encuesta de satisfacción (1 a 5)",
    "se_fue": "TARGET: 1 = se cambió de operador, 0 = se quedó",
    "particion": "Grupo train/test (no es variable del modelo)",
}

diccionario = pd.DataFrame({
    "descripcion": pd.Series(descripciones),
    "tipo_python": clientes.dtypes.astype(str),
    "valores_distintos": clientes.nunique(),
    "pct_nulos": (clientes.isnull().mean() * 100).round(1),
    "ejemplo": clientes.iloc[0],
})
diccionario["min_o_valores"] = [clientes[c].min() if pd.api.types.is_numeric_dtype(clientes[c]) else ", ".join(map(str, clientes[c].dropna().unique()[:4])) for c in diccionario.index]
diccionario["max"] = [clientes[c].max() if pd.api.types.is_numeric_dtype(clientes[c]) else "" for c in diccionario.index]
diccionario

### ✍️ Responde
Agrega en tu informe la columna **tipo de variable** (numérica continua, numérica discreta, categórica nominal, categórica ordinal, fecha, identificador). ¿Qué problemas de calidad revela el diccionario **antes** de limpiar? (mira `valores_distintos` de `departamento` y `tipo_plan`).

*Tu respuesta:*

### 3.3 Duplicados

In [ ]:
print("Filas duplicadas:", clientes.___().sum())        # TODO: método que detecta filas repetidas
datos = clientes.___().copy()                             # TODO: método que las elimina
print("Filas después de eliminarlos:", len(datos))
print("Clientes de prueba:", (datos["particion"] == "test").sum())

### 3.4 Inconsistencias en texto

In [ ]:
print(datos["departamento"].value_counts(), "\n")
print(datos["tipo_plan"].value_counts())

In [ ]:
# TODO: quita espacios y unifica mayúsculas (Mayúscula Inicial)
datos["departamento"] = datos["departamento"].str.___().str.___()
datos["tipo_plan"] = datos["tipo_plan"].str.___().str.___()

# TODO: después de unificar, todavía quedan valores escritos distinto. Corrígelos con replace
datos["departamento"] = datos["departamento"].replace({"___": "___"})
datos["tipo_plan"] = datos["tipo_plan"].replace({"___": "___"})

print(datos["departamento"].value_counts(), "\n")
print(datos["tipo_plan"].value_counts())

### 3.5 Valores imposibles
Una edad de 0 o de 999 años no es un atípico: es un **error** (no puede existir). Lo convertimos en nulo para tratarlo en el paso siguiente. Un monto negativo tampoco es posible.

In [ ]:
# TODO: ¿cuántas edades están fuera de un rango posible para un titular de línea (18 a 100)?
edad_imposible = (datos["edad"] < ___) | (datos["edad"] > ___)
print("Edades imposibles:", edad_imposible.sum(), "→", datos.loc[edad_imposible, "edad"].tolist())
datos.loc[edad_imposible, "edad"] = np.nan

print("Montos negativos:", (datos["monto_factura"] < 0).sum())
# TODO: decide qué hacer con ellos. Sugerencia: si el resto del registro es normal, fue un signo mal digitado
datos["monto_factura"] = datos["monto_factura"].___()

### 3.6 Atípicos en `monto_factura` (regla del IQR)
Recuerda: **atípico no es sinónimo de error**. Calcula los límites con los datos de `train` y luego investiga quiénes los superan.

In [ ]:
en_train = datos["particion"] == "train"
q1, q3 = datos.loc[en_train, "monto_factura"].quantile([0.25, 0.75])
iqr = ___                         # TODO
limite_sup = ___                  # TODO: Q3 + 1.5 · IQR
print(f"Q1 = {q1:.2f} | Q3 = {q3:.2f} | IQR = {iqr:.2f} | límite superior = {limite_sup:.2f}")

sobre_limite = datos[datos["monto_factura"] > limite_sup]
print("Clientes sobre el límite:", len(sobre_limite))
sobre_limite[["tipo_plan", "consumo_datos_mb", "monto_factura"]].sort_values("monto_factura")

### ✍️ Responde
Mira la tabla de los clientes sobre el límite. ¿Todos son errores? ¿Qué diferencia hay entre los que pagan entre 140 y 180 soles y los que pagan más de 2 000? Para los errores, ¿conviene **borrarlos**, **recortarlos** (capping) o **corregirlos**? Justifica y aplica tu decisión en la celda de abajo.

*Tu respuesta:*

In [ ]:
# TODO: aplica tu decisión sobre los montos con error (por ejemplo, los mayores a S/ 1 000)
errores = datos["monto_factura"] > ___
print("Montos corregidos:", errores.sum())
datos.loc[errores, "monto_factura"] = ___
print("Monto máximo ahora:", datos["monto_factura"].max())

### 3.7 Nulos
Rellena las numéricas con la **mediana** y las categóricas con la **moda**, calculadas **solo con train**.

In [ ]:
print("% de nulos:\n", (datos.isnull().mean() * 100).round(1)[lambda s: s > 0])

for col in ["edad", "monto_factura", "satisfaccion"]:
    mediana = datos.loc[en_train, col].___()          # TODO
    datos[col] = datos[col].fillna(mediana)
    print(f"{col}: relleno con la mediana {mediana}")

moda_pago = datos.loc[en_train, "metodo_pago"].___()[0]   # TODO
datos["metodo_pago"] = datos["metodo_pago"].fillna(moda_pago)
print("metodo_pago: relleno con la moda", moda_pago)
print("Nulos restantes:", datos.isnull().sum().sum())

### ✍️ Responde
`satisfaccion` tiene cerca de 11 % de nulos. ¿Qué efecto secundario tiene rellenar con la mediana? ¿Qué otra opción del árbol de decisión de nulos (sesión 3) podrías usar?

*Tu respuesta:*

### 3.8 Tipos de dato y discretización

In [ ]:
# TODO: convierte fecha_alta a fecha. Ojo: viene como día/mes/año
datos["fecha_alta"] = pd.to_datetime(datos["fecha_alta"], dayfirst=___)

# TODO: discretiza la edad en tramos
datos["tramo_edad"] = pd.cut(datos["edad"], bins=[17, 25, 40, 60, 100], labels=["18-25", "26-40", "41-60", "61+"])
print(datos.dtypes)
print(datos["tramo_edad"].value_counts().sort_index())

### ✍️ Responde · Resumen de calidad
Arma una tabla para tu informe: **problema → columna → cuántas filas afectó → decisión tomada**.

*Tu respuesta:*

## 4. Análisis exploratorio (sesión 3 · Tema 6)
### 4.1 Univariado: numéricas

In [ ]:
numericas = ["edad", "consumo_datos_mb", "minutos_llamadas", "recargas_mes", "monto_factura",
             "velocidad_mbps", "caidas_red_mes", "llamadas_soporte", "satisfaccion"]
display(datos[numericas].describe().round(1))

fig, ax = plt.subplots(3, 3, figsize=(14, 9))
for a, col in zip(ax.ravel(), numericas):
    a.hist(datos[col], bins=25, color="steelblue", edgecolor="white")
    a.set_title(col)
plt.suptitle("Distribución de las variables numéricas (datos limpios)")
plt.tight_layout(); plt.show()

print("Asimetría (skew):")
print(datos[numericas].skew().round(2).sort_values(ascending=False))

### 4.2 Univariado: categóricas y balance del target

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(18, 3.5))
datos["departamento"].value_counts().plot(kind="barh", ax=ax[0], color="steelblue"); ax[0].set_title("departamento")
datos["tipo_plan"].value_counts().plot(kind="bar", ax=ax[1], color="teal", rot=0); ax[1].set_title("tipo_plan")
datos["metodo_pago"].value_counts().plot(kind="bar", ax=ax[2], color="slateblue", rot=15); ax[2].set_title("metodo_pago")
datos["se_fue"].value_counts().sort_index().plot(kind="bar", ax=ax[3], color=["seagreen", "crimson"], rot=0)
ax[3].set_title("TARGET se_fue (0 = se queda, 1 = se va)")
plt.tight_layout(); plt.show()
print("% que se fue:", round(datos["se_fue"].mean() * 100, 1))

### ✍️ Responde
¿Qué variables tienen cola larga a la derecha? ¿Cómo está balanceado el target y qué "piso" de exactitud debe superar un modelo?

*Tu respuesta:*

### 4.3 Bivariado: numéricas vs target

In [ ]:
cols_box = ["llamadas_soporte", "satisfaccion", "caidas_red_mes", "consumo_datos_mb", "edad", "velocidad_mbps"]
fig, ax = plt.subplots(1, 6, figsize=(20, 3.6))
for a, col in zip(ax, cols_box):
    a.boxplot([datos.loc[datos["se_fue"] == 0, col], datos.loc[datos["se_fue"] == 1, col]])
    a.set_xticks([1, 2], ["se queda", "se va"]); a.set_title(col)
plt.tight_layout(); plt.show()
datos.groupby("se_fue")[cols_box].median()

### 4.4 Bivariado: categóricas vs target (tasa de fuga)

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(19, 3.6))
for a, col in zip(ax, ["tipo_plan", "metodo_pago", "departamento", "tramo_edad"]):
    tasa = datos.groupby(col, observed=True)["se_fue"].mean() * 100
    tasa.plot(kind="bar", ax=a, color="crimson", rot=20)
    a.set_title(f"% que se fue según {col}"); a.set_ylabel("%")
plt.tight_layout(); plt.show()

### 4.5 Correlación entre numéricas

In [ ]:
corr = datos[numericas + ["se_fue"]].corr()
plt.figure(figsize=(8, 6.5))
plt.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1); plt.colorbar()
plt.xticks(range(len(corr)), corr.columns, rotation=45, ha="right"); plt.yticks(range(len(corr)), corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        plt.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7)
plt.title("Matriz de correlación"); plt.tight_layout(); plt.show()

### ✍️ Responde
1. Escribe una conclusión (con la plantilla) para cada gráfico de 4.3 y 4.4.
2. ¿Qué variables **no** parecen relacionarse con la fuga?
3. ¿Qué pares de variables numéricas dicen casi lo mismo (redundancia)? Esto será importante para el PCA.

*Tu respuesta:*

### 4.6 Hipótesis
Formula **al menos 3 hipótesis** con el formato *"Los clientes que [condición] tienen [mayor/menor] [resultado] que los que [otra condición]"* y compruébalas con una tasa por grupo.

In [ ]:
tasa = lambda filtro: round(datos.loc[filtro, "se_fue"].mean() * 100, 1)

# Ejemplo resuelto
print("H1 · Prepago:", tasa(datos["tipo_plan"] == "Prepago"), "% | Postpago:", tasa(datos["tipo_plan"] != "Prepago"), "%")

# TODO H2: clientes con 2 o más llamadas a soporte vs con menos de 2
print("H2 · ...", tasa(___), "% | ...", tasa(___), "%")

# TODO H3: formula una hipótesis sobre el método de pago o la satisfacción
print("H3 · ...", tasa(___), "% | ...", tasa(___), "%")

# TODO H4: formula una hipótesis que creas que NO se cumplirá (p. ej., Lima vs regiones)
print("H4 · ...", tasa(___), "% | ...", tasa(___), "%")

## 5. Ingeniería de características (sesión 4 · Tema 7)
### 5.1 Crear nuevas features desde el negocio

In [ ]:
fecha_corte = pd.Timestamp("2026-10-31")

# TODO: antigüedad del cliente en meses (días entre fecha_corte y fecha_alta / 30.4)
datos["antiguedad_meses"] = ((fecha_corte - datos["fecha_alta"]).dt.days / ___).round()

# TODO: 1 si el cliente tiene menos de 6 meses, 0 si no
datos["cliente_nuevo"] = (datos["antiguedad_meses"] < ___).astype(int)

# TODO: 1 si hizo 2 o más llamadas a soporte
datos["soporte_frecuente"] = (___).astype(int)

# TODO: crea UNA feature propia y justifícala (ej.: soles por GB, caídas por cada llamada a soporte...)
datos["mi_feature"] = ___

datos[["antiguedad_meses", "cliente_nuevo", "soporte_frecuente", "mi_feature"]].describe().round(2)

### 5.2 Re-escalado no lineal: logaritmo
Aplica `log1p` a la variable con la cola más larga y compara su asimetría antes y después.

In [ ]:
datos["log_datos"] = np.log1p(datos["___"])        # TODO

fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].hist(datos["consumo_datos_mb"], bins=30, color="steelblue"); ax[0].set_title("consumo_datos_mb (original)")
ax[1].hist(datos["log_datos"], bins=30, color="darkorange"); ax[1].set_title("log(1 + MB)")
plt.tight_layout(); plt.show()
print("Asimetría antes:", round(datos["consumo_datos_mb"].skew(), 2), "| después:", round(datos["log_datos"].skew(), 2))

### 5.3 Rehacer el EDA con las features nuevas

In [ ]:
datos["tramo_antiguedad"] = pd.cut(datos["antiguedad_meses"], bins=[-1, 6, 12, 24, 48, 200],
                                   labels=["0-6", "7-12", "13-24", "25-48", "49+"])
t = datos.groupby("tramo_antiguedad", observed=True)["se_fue"].mean() * 100
t.plot(kind="bar", color="crimson", rot=0, figsize=(6, 3.4))
plt.ylabel("% que se fue"); plt.xlabel("meses de antigüedad"); plt.title("Fuga según antigüedad del cliente"); plt.show()
print(t.round(1))

### ✍️ Responde
¿Qué muestra la antigüedad? Recuerda que esta información estaba "escondida" en una columna de texto que el escenario A no pudo usar.

*Tu respuesta:*

### 5.4 Codificar categóricas (one-hot)

In [ ]:
datos_mod = pd.get_dummies(datos, columns=["departamento", "tipo_plan", "metodo_pago"], drop_first=___, dtype=int)   # TODO
print("Columnas:", datos_mod.shape[1])
datos_mod.filter(regex="tipo_plan_|metodo_pago_").head()

### 5.5 Selección de características: información mutua
Calcúlala **solo con train**. Las variables 0/1 se marcan como discretas.

In [ ]:
no_son_features = ["id_cliente", "fecha_alta", "se_fue", "particion", "tramo_edad", "tramo_antiguedad"]
candidatas = [c for c in datos_mod.columns if c not in no_son_features]
en_train = datos_mod["particion"] == "train"

discretas = [datos_mod[c].nunique() <= 2 for c in candidatas]
mi = mutual_info_classif(datos_mod.loc[en_train, candidatas], datos_mod.loc[en_train, "se_fue"],
                         discrete_features=discretas, random_state=42)
mi = pd.Series(mi, index=candidatas).sort_values()

plt.figure(figsize=(7, 7)); mi.plot(kind="barh", color="teal")
plt.title("Información mutua con se_fue (train)"); plt.xlabel("información mutua"); plt.show()
print(mi.sort_values(ascending=False).round(3))

Ahora **decide** qué variables entran al modelo. Usa la información mutua **y** lo que viste en el EDA (no solo el ranking). Justifica cada exclusión.

In [ ]:
# TODO: arma la lista de variables que entrarán a los escenarios B y C
features = [
    ___,
]
print(len(features), "variables:", features)

### 5.6 Re-escalado lineal: estandarización
El `fit` del escalador se hace **solo con train**; luego se aplica (`transform`) a train y a test.

In [ ]:
X_train = datos_mod.loc[en_train, features]
X_test = datos_mod.loc[~en_train, features]
y_train = datos_mod.loc[en_train, "se_fue"]
y_test = datos_mod.loc[~en_train, "se_fue"]

escalador = StandardScaler()
X_train_z = escalador.___(X_train)     # TODO: aprende y aplica en train
X_test_z = escalador.___(X_test)       # TODO: SOLO aplica en test

print("Train:", X_train_z.shape, "| Test:", X_test_z.shape)
print("Medias en train (≈ 0):", X_train_z.mean(axis=0).round(2)[:5])

## 6. Escenario B · El modelo CON estrategias
Mismo modelo, mismos 300 clientes de prueba. Solo cambió la preparación de los datos.

In [ ]:
modelo_B = evaluar_modelo(X_train_z, y_train, X_test_z, y_test, "B · Con estrategias")

In [ ]:
# ¿Qué variables pesan más en el modelo B? (coeficientes de la regresión logística con datos estandarizados)
pesos = pd.Series(modelo_B.coef_[0], index=features).sort_values()
pesos.plot(kind="barh", color=np.where(pesos > 0, "crimson", "seagreen"), figsize=(7, 5))
plt.title("Peso de cada variable en el modelo B\n(rojo = aumenta la fuga, verde = la reduce)"); plt.show()

### ✍️ Responde (escenario B)
1. Compara con A: ¿cuántos clientes que se iban detecta ahora el modelo (VP)? ¿Cuántos FN menos hay?
2. ¿Apareció el `ConvergenceWarning`? ¿Por qué?
3. ¿Las variables con más peso coinciden con tus hipótesis del EDA?

*Tu respuesta:*

## 7. Extracción de características: PCA (sesión 4 · Tema 7.2)
### 7.1 ¿Cuánta varianza conserva cada componente?
Los datos ya están estandarizados (paso obligatorio antes de PCA). El `fit` del PCA también se hace **solo con train**.

In [ ]:
pca_completo = PCA().fit(X_train_z)
var = pca_completo.explained_variance_ratio_
acum = var.cumsum()

plt.figure(figsize=(8, 3.6))
plt.bar(range(1, len(var) + 1), var * 100, color="steelblue", label="cada componente")
plt.plot(range(1, len(var) + 1), acum * 100, "o-", color="darkorange", label="acumulada")
plt.axhline(90, ls="--", color="gray"); plt.text(len(var) - 2, 92, "90 %")
plt.xlabel("componente"); plt.ylabel("% de varianza"); plt.title("Varianza explicada por el PCA (train)")
plt.legend(); plt.show()

print(pd.DataFrame({"var_%": (var * 100).round(1), "acumulada_%": (acum * 100).round(1)}, index=[f"PC{i}" for i in range(1, len(var) + 1)]))

### ✍️ Responde
¿Cuánta varianza conservan 2 componentes? ¿Cuántas componentes hacen falta para llegar al 90 %? Comparado con el número de variables originales, ¿cuánto se reduce?

*Tu respuesta:*

### 7.2 Ver los datos en 2D

In [ ]:
pca2 = PCA(n_components=2).fit(X_train_z)
comp = pca2.transform(X_train_z)
plt.figure(figsize=(7, 5))
for clase, color, etiqueta in [(0, "seagreen", "se queda"), (1, "crimson", "se va")]:
    m = (y_train == clase).values
    plt.scatter(comp[m, 0], comp[m, 1], s=10, alpha=0.5, c=color, label=etiqueta)
plt.xlabel(f"PC1 ({pca2.explained_variance_ratio_[0]*100:.1f} %)"); plt.ylabel(f"PC2 ({pca2.explained_variance_ratio_[1]*100:.1f} %)")
plt.title("Clientes de train proyectados en 2 componentes"); plt.legend(); plt.show()

# ¿Qué variables forman cada componente? (cargas)
cargas = pd.DataFrame(pca2.components_.T, index=features, columns=["PC1", "PC2"]).round(2)
cargas.sort_values("PC1")

### ✍️ Responde
Mira las cargas: ¿qué variables pesan más en la PC1? ¿Qué "idea de negocio" resume esa componente? En el gráfico 2D, ¿se separan bien los que se van de los que se quedan?

*Tu respuesta:*

### 7.3 Escenario C · Estrategias + PCA
Pasamos al modelo las componentes que conservan el **90 %** de la varianza (`n_components=0.90`).

In [ ]:
pca = PCA(n_components=___, random_state=42)     # TODO: 0.90 = conservar el 90 % de la varianza
X_train_pca = pca.___(X_train_z)                 # TODO: fit solo con train
X_test_pca = pca.___(X_test_z)                   # TODO
print("Componentes usadas:", pca.n_components_, "de", len(features), "variables")

modelo_C = evaluar_modelo(X_train_pca, y_train, X_test_pca, y_test, "C · Estrategias + PCA")

**Experimento extra:** ¿qué pasa si el modelo solo recibe **2 componentes** (las del gráfico 2D)?

In [ ]:
X_train_2 = pca2.transform(X_train_z)
X_test_2 = pca2.transform(X_test_z)
modelo_C2 = evaluar_modelo(X_train_2, y_train, X_test_2, y_test, "C2 · Solo 2 componentes")

## 8. Comparación final de las matrices de confusión

In [ ]:
fig, ax = plt.subplots(1, len(matrices), figsize=(5 * len(matrices), 4.2))
for a, (nombre, mc) in zip(ax, matrices.items()):
    ConfusionMatrixDisplay(mc, display_labels=["Se queda", "Se va"]).plot(ax=a, cmap="Blues", colorbar=False)
    a.set_title(nombre); a.set_xlabel("Predicción"); a.set_ylabel("Realidad")
plt.suptitle("Mismo modelo, mismos 300 clientes de prueba: solo cambia la preparación de los datos", y=1.02)
plt.tight_layout(); plt.show()

tabla = pd.DataFrame(resultados).T
tabla

In [ ]:
tabla[["exactitud", "precision", "recall", "f1"]].plot(kind="bar", rot=0, figsize=(10, 4),
                                                       color=["steelblue", "slateblue", "crimson", "darkorange"])
plt.ylim(0, 1); plt.title("Métricas por escenario"); plt.legend(loc="lower right"); plt.show()

### ✍️ Responde · Conclusión de la práctica
1. **A vs B:** ¿cuántos clientes en fuga más detecta B? Si retener a cada cliente vale S/ 600 al año, ¿cuánto dinero representa esa diferencia en estos 300 clientes?
2. ¿Qué estrategia crees que aportó **más** a la mejora: la limpieza, las features nuevas (antigüedad, tipo de plan), o el escalado? ¿Cómo lo comprobarías? (Pista: prueba quitando un paso a la vez.)
3. **B vs C:** ¿el PCA mejoró, empató o empeoró la matriz? ¿Qué ganaste y qué perdiste al usarlo (número de variables, interpretabilidad)?
4. **C vs C2:** ¿por qué con solo 2 componentes el modelo empeora? Relaciónalo con el % de varianza que conservan.
5. Para este proyecto, ¿recomendarías usar PCA? Justifica con lo visto en la diapositiva *"Selección vs extracción"*.

*Tu respuesta:*

## 9. Reproducibilidad
- Ejecuta `Entorno de ejecución → Reiniciar y ejecutar todo` y verifica que no haya errores.
- Guarda el dataset final que usarías en el bloque 3:

In [ ]:
datos_mod[features + ["se_fue", "particion"]].to_csv("clientes_telecom_final.csv", index=False)
import sklearn
print("pandas", pd.__version__, "| scikit-learn", sklearn.__version__)

## Reto opcional (para ir más allá)
1. El target está desbalanceado. Repite el escenario B con `LogisticRegression(max_iter=1000, class_weight="balanced")` (cambia la línea dentro de `evaluar_modelo`). ¿Qué pasa con los FN y con los FP? ¿Le conviene a la operadora?
2. Cambia la regresión logística por `KNeighborsClassifier(n_neighbors=7)` (de `sklearn.neighbors`) y repite A y B. ¿Por qué un modelo basado en **distancias** sufre todavía más sin escalar?